# CrewAI Demo 1: Workshop Planning Team

This demo introduces CrewAI using a simple but realistic planning problem.

A company wants to run a short AI workshop for non-technical managers. The work naturally breaks into several responsibilities:

- curriculum design
- logistics
- budget planning
- final coordination

This is a good fit for CrewAI because CrewAI is organized around agents, tasks, crews, and execution processes. In this demo, each agent owns a specialized task and hands its output to the next task.

The goal is not to create a debate or open-ended conversation.  
The goal is to show how a small team of specialized agents can produce a shared deliverable.


## Why CrewAI Fits This Problem

This workshop-planning problem naturally decomposes into specialized responsibilities:

- The curriculum specialist designs the learning experience.
- The logistics specialist makes the plan executable.
- The budget specialist checks cost and resource constraints.
- The coordinator assembles the final plan.

That makes CrewAI a natural fit because:
- agents have stable roles
- tasks have clear ownership
- handoffs are predictable
- the final deliverable is structured

Other frameworks could also solve this problem.

AutoGen would emphasize conversation between agents. That is useful when interaction, debate, or critique is the center of the demo.

LangGraph would emphasize workflow state, routing, branching, and orchestration. That is useful when control flow is the center of the demo.

Here, the main idea is simpler:

> A small team of specialists completes delegated tasks and assembles a shared work product.

That is the CrewAI pattern we want to teach first.


## What This Demo Teaches

This intro demo focuses on the basic CrewAI mental model:

1. Define agents with roles, goals, and backstories.
2. Define tasks with clear expected outputs.
3. Put agents and tasks into a crew.
4. Run the crew using a sequential process.
5. Inspect each task output so the handoffs are visible.

This demo is intentionally one pass.

The later bike-shop demo adds a second refinement cycle, stronger constraints, and more realistic tradeoffs.


## Scenario

A company wants to run a 2-hour introduction to AI workshop for non-technical managers.

Requirements:

- The workshop should be practical and interactive.
- Avoid heavy technical jargon.
- Include at least one hands-on exercise.
- Support both in-room and remote attendees.
- Keep the total budget under $1,500.
- Produce a final plan that could be given to an event organizer.

The final output should include:

- workshop agenda
- learning objectives
- logistics checklist
- budget summary
- facilitator preparation notes


In [1]:
# Install packages if needed:
# pip install crewai crewai-tools
# pip install python-dotenv

import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

from crewai import Agent, Task, Crew, Process

load_dotenv()

# CrewAI typically reads OPENAI_API_KEY from the environment.
# Do NOT hardcode API keys in notebooks.
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your environment or .env file."


In [2]:
WORKSHOP_SCENARIO = """
A company wants to run a 2-hour introduction to AI workshop for non-technical managers.

Requirements:
- The workshop should be practical and interactive.
- Avoid heavy technical jargon.
- Include at least one hands-on exercise.
- Support both in-room and remote attendees.
- Keep the total budget under $1,500.
- Produce a final plan that could be given to an event organizer.

The final output should include:
- workshop agenda
- learning objectives
- logistics checklist
- budget summary
- facilitator preparation notes
"""

MODEL_NAME = "gpt-4.1-mini"


## Helper Display Functions

CrewAI's verbose output is useful for debugging, but it can be noisy in a classroom notebook.

Here we turn verbose logging off and print the important parts ourselves:

- agent role
- task description
- task output

This makes the handoffs easier to see.


In [3]:
def shorten(text, max_len=600):
    text = str(text).strip()
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."


def task_output_text(task):
    # CrewAI versions vary slightly in how task output is represented.
    # This helper keeps the notebook tolerant across versions.
    output = getattr(task, "output", None)

    if output is None:
        return "[No output found. Did you run crew.kickoff()?]"

    raw = getattr(output, "raw", None)
    if raw:
        return raw

    return str(output)


def show_task(title, task, show_description=True):
    agent_role = getattr(task.agent, "role", "Unknown agent")

    task_section = ""
    if show_description:
        task_section = f"\n## Task\n{shorten(task.description)}\n"

    display(Markdown(f"""
# {title}

## Agent
**{agent_role}**
{task_section}
## Output
{task_output_text(task)}
"""))


## Single-Agent Baseline

Before using a crew, we create a simple one-agent baseline.

This gives us something to compare against.

The baseline may be good. The point is not to make single-agent prompting look bad.

The question is whether a specialized crew gives us:
- clearer task ownership
- better coverage
- more complete planning
- more useful final deliverable


In [4]:
baseline_agent = Agent(
    role="Workshop Planner",
    goal="Create a practical AI workshop plan for non-technical managers.",
    backstory=(
        "You design practical workshops for business audiences. "
        "You avoid jargon and focus on useful, interactive learning."
    ),
    llm=MODEL_NAME,
    verbose=False,
)

baseline_task = Task(
    description=f"""
Scenario:

{WORKSHOP_SCENARIO}

Create a complete 2-hour workshop plan.
Include agenda, learning objectives, logistics, budget, and facilitator preparation notes.
""",
    expected_output=(
        "A complete 2-hour workshop plan for non-technical managers, including agenda, "
        "learning objectives, logistics, budget, and facilitator preparation notes."
    ),
    agent=baseline_agent,
)

baseline_crew = Crew(
    agents=[baseline_agent],
    tasks=[baseline_task],
    process=Process.sequential,
    verbose=False,
)

baseline_result = baseline_crew.kickoff()

show_task("Single-Agent Baseline", baseline_task, show_description=False)



# Single-Agent Baseline

## Agent
**Workshop Planner**

## Output
**Workshop Plan: Introduction to AI for Non-Technical Managers**

---

## Workshop Title:
**AI Essentials: Practical Insights for Non-Technical Managers**

---

## Workshop Duration:
2 Hours

---

## Workshop Overview:
This workshop introduces non-technical managers to fundamental AI concepts, demystifies AI's role in business, and provides an interactive exercise to practically engage with AI tools relevant to decision-making and team leadership.

---

## Learning Objectives:

By the end of this workshop, participants will be able to:

1. Understand what AI is and recognize common AI applications in business contexts.
2. Identify opportunities where AI can enhance workflows and decision-making.
3. Appreciate the limitations and ethical considerations of AI.
4. Confidently discuss AI concepts and initiatives with technical teams.
5. Apply a simple AI-based tool through a hands-on group exercise to solve a business problem.

---

## Workshop Agenda:

| Time      | Topic/Activity                           | Details                                                                | Format                  |
|-----------|----------------------------------------|------------------------------------------------------------------------|-------------------------|
| 0:00–0:10| Welcome and Introductions               | Facilitator introduction, participant introductions, workshop goals    | Interactive discussion  |
| 0:10–0:25| What is AI?                            | Simple definitions, real-world examples, dispelling myths              | Presentation + Q&A       |
| 0:25–0:40| AI Use Cases in Business               | Examples across industries, focus on manager-relevant scenarios        | Presentation + group chat|
| 0:40–0:55| AI Limitations and Ethics              | Understand bias, data privacy, AI pitfalls                             | Presentation + open discussion|
| 0:55–1:10| Break                                |                                                                      | —                       |
| 1:10–1:50| Hands-on Exercise: Using AI Tools to Solve a Problem | Participants work in small groups (in-room and remote breakout rooms) on a simple AI-related scenario (e.g., using AI-powered text summarization or sentiment analysis tool on sample business data). Facilitator available for support. | Group exercise; live demo; group discussion |
| 1:50–2:00| Wrap-up and Q&A                      | Summarize key takeaways, answer questions, provide resources           | Discussion               |

---

## Hands-On Exercise Details:

- **Objective:** Use a user-friendly AI tool (e.g., a web-based AI text analyzer or chatbot) to analyze short text data (e.g., customer feedback or internal emails) and extract insights.
- **Tools:** Free or trial-based web tools (e.g., MonkeyLearn, IBM Watson Discovery Lite, or Google Cloud Natural Language demo).
- **Setup:** 
  - Provide participants with access links beforehand.
  - Prepare sample text datasets.
  - Guide participants step-by-step on how to input data and interpret outputs.
- **Outcome:** Experience firsthand how AI can timely analyze data and support business decisions.

---

## Logistics Checklist:

| Item                       | Details/Notes                                                             | Responsible      |
|----------------------------|--------------------------------------------------------------------------|------------------|
| Venue setup                | Room with projector, whiteboard, breakout spaces for small groups        | Event Organizer  |
| AV Equipment               | Projector, microphone, speakers, camera for remote live stream            | Event Organizer  |
| Wi-Fi                      | Stable internet connection with guest access                             | Event Organizer  |
| Participant devices        | Participants bring laptops/tablets or provide a limited number of devices| Participants / Organizer|
| Remote participation setup | Video conferencing platform (Zoom/Microsoft Teams) with breakout rooms   | Facilitator & Organizer|
| AI tool access             | Pre-register/free trial links shared with participants                    | Facilitator      |
| Printed materials          | Agenda printouts, instruction sheets for exercise                        | Organizer        |
| Refreshments               | Optional, within budget                                                  | Organizer        |
| Registration and reminders | Email invitations, reminders with links and instructions                 | Organizer        |

---

## Budget Summary (Estimate):

| Item                              | Estimated Cost  | Notes                           |
|----------------------------------|-----------------|--------------------------------|
| Venue rental                     | $0 - $300       | Use company space if available  |
| AV Equipment                     | $0              | Use existing equipment          |
| Facilitator fee                  | $800            | Includes preparation and delivery |
| Printed materials               | $50             | Handouts and worksheets         |
| Refreshments                    | $150            | Coffee, water, light snacks     |
| Remote platform subscription    | $0 - $100       | Use existing or trial accounts  |
| Miscellaneous                  | $100            | Contingency                    |
| **Total Estimated Cost**        | **$1,100 - $1,500** | Keep budget flexibility         |

*Note: Cost savings possible by using internal resources and existing technology.*

---

## Facilitator Preparation Notes:

- **Pre-Workshop:**
  - Familiarize with all AI tools to be demonstrated.
  - Test connectivity and remote participation tools.
  - Prepare clear, jargon-free slides and materials.
  - Prepare example datasets for the exercise.
  - Send pre-workshop instructions and materials at least 3 days in advance.
  
- **During Workshop:**
  - Engage participants by encouraging questions and sharing relatable examples.
  - Monitor time strictly to allow full coverage of agenda.
  - Facilitate breakout rooms ensuring remote and in-person groups are supported.
  - Provide clear, simple instructions for the hands-on exercise.
  - Be ready to troubleshoot technical issues quickly.
  
- **Post-Workshop:**
  - Share session recording (if applicable), slides, and additional learning resources.
  - Provide a follow-up email with AI tool links and recommended readings.
  - Collect feedback for continual improvement.

---

# End of Workshop Plan

---

**This plan can be directly handed to the event organizer for implementation.**


## Define the Crew

Now we define a small team.

Each agent owns a different responsibility:

- CurriculumAgent designs the learning experience.
- LogisticsAgent makes the event executable.
- BudgetAgent keeps the plan within constraints.
- CoordinatorAgent assembles the final deliverable.

This is the simplest useful CrewAI pattern:

> role specialization + task handoff + final synthesis


In [5]:
curriculum_agent = Agent(
    role="Curriculum Designer",
    goal="Design a clear, practical, interactive AI workshop agenda.",
    backstory=(
        "You design workshops for adult learners. "
        "You specialize in turning technical topics into practical exercises for non-technical audiences."
    ),
    llm=MODEL_NAME,
    verbose=False,
    allow_delegation=False,
)

logistics_agent = Agent(
    role="Logistics Coordinator",
    goal="Make the workshop plan executable for both in-room and remote attendees.",
    backstory=(
        "You coordinate training events and hybrid meetings. "
        "You focus on room setup, timing, materials, remote access, and attendee experience."
    ),
    llm=MODEL_NAME,
    verbose=False,
    allow_delegation=False,
)

budget_agent = Agent(
    role="Budget Analyst",
    goal="Keep the workshop plan realistic and within the $1,500 budget.",
    backstory=(
        "You help teams deliver practical events on limited budgets. "
        "You identify cost risks, savings opportunities, and sensible contingency reserves."
    ),
    llm=MODEL_NAME,
    verbose=False,
    allow_delegation=False,
)

coordinator_agent = Agent(
    role="Workshop Coordinator",
    goal="Synthesize specialist inputs into a complete, ready-to-use workshop plan.",
    backstory=(
        "You turn partial plans from specialists into practical final deliverables. "
        "You are concise, organized, and focused on execution."
    ),
    llm=MODEL_NAME,
    verbose=False,
    allow_delegation=False,
)


## Define the Tasks

The tasks are intentionally simple and ordered.

This is what makes the first demo easy to understand:

1. Curriculum creates the agenda.
2. Logistics makes the agenda executable.
3. Budget checks the plan against the spending limit.
4. Coordinator assembles the final workshop plan.

CrewAI's sequential process runs tasks one after another in the order they are listed.


In [6]:
curriculum_task = Task(
    description=f"""
Scenario:

{WORKSHOP_SCENARIO}

Design the learning experience for the workshop.

Your output must include:
1. Learning objectives
2. A 2-hour agenda with time blocks
3. At least one hands-on exercise
4. Discussion prompts for managers
5. Suggested plain-language explanations for key AI concepts

Keep the content practical and non-technical.
""",
    expected_output=(
        "A curriculum plan with objectives, timed agenda, hands-on exercise, "
        "discussion prompts, and plain-language explanations."
    ),
    agent=curriculum_agent,
)

logistics_task = Task(
    description="""
Review the curriculum plan from the previous task.

Create a logistics plan that supports both in-room and remote attendees.

Your output must include:
1. Room or virtual setup
2. Materials needed
3. Timing and facilitation notes
4. Remote attendee support
5. Risks or setup issues to avoid

Do not redesign the curriculum.
Make the existing plan executable.
""",
    expected_output=(
        "A logistics plan for delivering the workshop in a hybrid setting."
    ),
    agent=logistics_agent,
    context=[curriculum_task],
)

budget_task = Task(
    description="""
Review the curriculum and logistics plans.

Create a budget plan under $1,500.

Your output must include:
1. Estimated cost categories
2. What to spend money on
3. What to avoid spending money on
4. Contingency reserve
5. Any changes needed to keep the plan affordable

Do not create a new workshop.
Constrain and improve the existing plan financially.
""",
    expected_output=(
        "A budget plan under $1,500 with cost categories, tradeoffs, and contingency."
    ),
    agent=budget_agent,
    context=[curriculum_task, logistics_task],
)

final_plan_task = Task(
    description="""
Use the curriculum, logistics, and budget work to create the final workshop plan.

Your output must include:
1. Workshop title
2. Learning objectives
3. Final 2-hour agenda
4. Hands-on exercise
5. Logistics checklist
6. Budget summary
7. Facilitator preparation checklist

Make the final plan clear enough that an event organizer could use it.
""",
    expected_output=(
        "A complete ready-to-use workshop plan with agenda, logistics, budget, and facilitator checklist."
    ),
    agent=coordinator_agent,
    context=[curriculum_task, logistics_task, budget_task],
)


In [7]:
workshop_crew = Crew(
    agents=[
        curriculum_agent,
        logistics_agent,
        budget_agent,
        coordinator_agent,
    ],
    tasks=[
        curriculum_task,
        logistics_task,
        budget_task,
        final_plan_task,
    ],
    process=Process.sequential,
    verbose=False,
)

crew_result = workshop_crew.kickoff()


## Inspect the Handoffs

Instead of printing raw verbose logs, we inspect each task output in workflow order.

This is the key part of the demo.

Students should see how each specialist contributes a different work product.


In [8]:
show_task("1. Curriculum Plan", curriculum_task)
show_task("2. Logistics Plan", logistics_task)
show_task("3. Budget Plan", budget_task)
show_task("4. Final Coordinated Workshop Plan", final_plan_task)



# 1. Curriculum Plan

## Agent
**Curriculum Designer**

## Task
Scenario:


A company wants to run a 2-hour introduction to AI workshop for non-technical managers.

Requirements:
- The workshop should be practical and interactive.
- Avoid heavy technical jargon.
- Include at least one hands-on exercise.
- Support both in-room and remote attendees.
- Keep the total budget under $1,500.
- Produce a final plan that could be given to an event organizer.

The final output should include:
- workshop agenda
- learning objectives
- logistics checklist
- budget summary
- facilitator preparation notes


Design the learning experience for the workshop.

Your output m...

## Output
**Workshop Title:**  
Demystifying AI for Managers: Practical Insights and Hands-On Experience

---

### 1. Learning Objectives

By the end of this 2-hour workshop, participants will be able to:  
1. Define Artificial Intelligence (AI) in simple, practical terms.  
2. Recognize common AI applications relevant to their business context.  
3. Understand key AI concepts without technical jargon (e.g., machine learning, data training).  
4. Identify opportunities and challenges related to AI in management decisions.  
5. Collaborate through an interactive exercise that simulates AI-powered decision-making.  
6. Engage in meaningful conversations about integrating AI thoughtfully into their teams and workflows.

---

### 2. Workshop Agenda (2 Hours Total)

| Time           | Activity                                     | Format                  | Notes                              |
|----------------|----------------------------------------------|-------------------------|------------------------------------|
| 0:00 - 0:10    | **Welcome & Introductions**                   | Group (In-person + virtual)       | Establish rapport; quick round of introductions with name + one expectation |
| 0:10 - 0:25    | **What is AI? – Plain Language Overview**    | Facilitator Presentation + Q&A    | Use analogies; avoid jargon         |
| 0:25 - 0:40    | **Real-World AI Examples**                     | Group Discussion + Slide Examples | Focus on familiar industries/tasks  |
| 0:40 - 1:00    | **Key AI Concepts Explained Simply**          | Facilitator Explanation            | Concepts: Machine Learning, Data Training, Algorithms, Automation |
| 1:00 - 1:10    | **Break**                                     |                         | Optional for remote attendees to stretch |
| 1:10 - 1:40    | **Hands-On Interactive Exercise:**  
*"AI in Action – Decision Simulation"*            | Small Groups / Breakouts           | See Exercise details below          |
| 1:40 - 1:55    | **Group Debrief & Discussion**                 | Whole Group Discussion            | Insights from exercise & reflections |
| 1:55 - 2:00    | **Wrap-Up, Q&A, and Next Steps**                | Facilitator-led                   | Provide resources for further learning |

---

### 3. Hands-On Exercise  

**Title:** AI in Action – Decision Simulation  

**Objective:** To experience how AI systems use data to make recommendations and understand implications for decision-making.

**Setup:**  
- Divide participants into small groups (3-5 people). For remote attendees, use breakout rooms in video conferencing platform.  
- Facilitator provides each group with a simple dataset (printed or shared digitally) about customer preferences or employee feedback.  
- Groups are given a basic "decision challenge" (e.g., choosing a marketing channel, assigning tasks, or hiring prioritization).  

**Instructions:**  
1. Each group reviews the data provided.  
2. Groups simulate “training” an AI by identifying key patterns or trends in the data that would inform their decision.  
3. Groups generate a recommendation based on the data patterns.  
4. Groups consider potential biases or gaps in the data affecting their decision.  

**Materials:**  
- Sample dataset: (e.g., table with customer age, purchase history, satisfaction rating)  
- Worksheet guiding the steps (simple prompts)  
- Timer  

**Facilitator Role:** Guide discussions, encourage teams to think about how an AI “learns” from data, and to reflect on human oversight.  

---

### 4. Discussion Prompts for Managers  

- How could AI tools help streamline decisions in your department or team?  
- What risks or challenges do you see with relying on AI-generated recommendations?  
- How can managers ensure ethical and fair use of AI within their teams?  
- What kind of data do you currently collect that might be useful for AI applications?  
- How can you foster collaboration between technical AI teams and non-technical managers?  

---

### 5. Suggested Plain-Language Explanations for Key AI Concepts  

- **Artificial Intelligence (AI):** Computers or software that can perform tasks that usually require human thinking, like recognizing patterns, making recommendations, or understanding language.  
- **Machine Learning:** A way computers improve their performance by learning from data, similar to how people learn from experience.  
- **Data Training:** Feeding large amounts of information to AI so it can recognize what’s important and make smarter decisions.  
- **Algorithms:** Step-by-step instructions or rules that the AI follows to analyze data and produce results. Think of this as a recipe for making decisions.  
- **Automation:** Using technology to do routine or repetitive tasks automatically, freeing up people to focus on more complex work.  
- **Bias:** When the AI’s decisions are unfair or limited because the data it learned from wasn’t complete or was one-sided.  

---

# Additional Workshop Components for Event Organizer Use

---

### Logistics Checklist

- **Venue Setup:**  
  - Room arranged with small group tables for breakout activities (in-person).  
  - Projector and screen or large monitor visible to all participants.  
  - Strong WiFi connection for remote attendees.  
  - Microphone and speaker system if room is large.  

- **Technology:**  
  - Video conferencing platform supporting breakout rooms (Zoom, MS Teams, etc.).  
  - Pre-upload all presentation materials and datasets to shared drive.  
  - Ensure screen sharing capabilities for facilitator.  

- **Materials:**  
  - Printed or digital copies of data sets and worksheets for each group.  
  - Name tags (for in-person).  
  - Timer or stopwatch app.  

- **Facilitator Requirements:**  
  - Laptop with presentation slides.  
  - Access to video conferencing and breakout room controls.  
  - Backup plan for technical hiccups (e.g., sharing materials via chat).  

---

### Budget Summary (Estimate)

| Item                            | Cost Estimate          | Notes                                  |
|--------------------------------|-----------------------|---------------------------------------|
| Venue Rental (if applicable)    | $300                  | Assume small meeting room              |
| AV Equipment Rental             | $200                  | Projector, mic, speakers (if not included in venue) |
| Video Conferencing Platform     | Free or $100           | Use existing license or free plan with breakout rooms |
| Printing of Handouts/Datasets   | $50                   | 20 participants x 3 pages             |
| Facilitator Fee                 | $600                  | Workshop design + delivery             |
| Misc (Refreshments, Misc. Supplies) | $150           | Coffee, snacks, name tags etc.         |
| Contingency                    | $100                  | Unexpected expenses                    |
| **Total Estimated Budget**      | **$1,500**             |                                       |

*Note:* Costs can vary depending on location or virtual-only meeting (which reduces venue and AV costs).

---

### Facilitator Preparation Notes

- Familiarize yourself with the audience’s business context to give relevant examples.  
- Practice explaining AI concepts in clear, everyday language; avoid acronyms and jargon.  
- Prepare to monitor and support breakout rooms actively during the exercise, ensuring engagement and clarity.  
- Anticipate questions about risks and ethical concerns; prepare responses that acknowledge the complexity but focus on practical steps.  
- Have backup activities ready in case of technical difficulties with breakout rooms.  
- Encourage participants to share stories and concerns to make the session more interactive.  
- Review timing carefully; keep pace to allow group discussions and reflection time.  

---

**End of Workshop Plan**



# 2. Logistics Plan

## Agent
**Logistics Coordinator**

## Task
Review the curriculum plan from the previous task.

Create a logistics plan that supports both in-room and remote attendees.

Your output must include:
1. Room or virtual setup
2. Materials needed
3. Timing and facilitation notes
4. Remote attendee support
5. Risks or setup issues to avoid

Do not redesign the curriculum.
Make the existing plan executable.

## Output
**Logistics Plan for Hybrid Delivery**  
**Workshop Title:** Demystifying AI for Managers: Practical Insights and Hands-On Experience  
**Duration:** 2 hours  

---

### 1. Room and Virtual Setup

**In-Room Setup:**  
- Arrange the physical room with small group tables seating 3-5 participants each to enable breakout activities and discussions.  
- Position a projector and screen or large monitor visible to all in-person attendees to display slides, timer, and facilitator’s video feed.  
- Ensure strong, stable WiFi connection supporting seamless video conferencing connection (upload/download speed sufficient for streaming and breakout rooms).  
- Set up a microphone and speaker system if the room is large enough that voices do not carry well, enabling clear audio for remote attendees and facilitator.  
- Reserve a space near the facilitator for computer and AV control (projector, screen sharing).  
- Place name tags and printed materials on tables prior to participant arrival.  

**Virtual Setup:**  
- Use a video conferencing platform supporting breakout rooms, screen sharing, chat, and reactions: recommended Zoom, MS Teams, or equivalent.  
- Pre-configure breakout rooms for the hands-on exercise; facilitator should have controls to monitor and join rooms as needed.  
- Pre-upload all presentation slides, datasets, and worksheets to a shared drive or platform (e.g., Google Drive, SharePoint) with participant access links shared in the chat during the session.  
- Open a main meeting room for the full group and breakout spaces for small group exercises.  
- Enable participant video and audio for engagement but set to mute upon entry to avoid background noise.  
- Prepare a designated technical support contact (can be a co-host) available throughout the session to assist remote attendees with connection or tech issues.  

---

### 2. Materials Needed

**For In-Person Participants:**  
- Printed workshop agenda and timings (1 per participant, visible on tables).  
- Printed copies of:  
  - Sample dataset sheet (one per small group).  
  - Hands-on exercise worksheet guiding steps (one per small group).  
  - Discussion prompts sheet (optional, 1 per participant for reflection).  
- Name tags with clear names and roles (if possible).  
- Timer or stopwatch device (physical or app) visible to facilitator and ideally positioned where all groups can see it, or announced verbally during time checks.  
- Pens and notepads for notes.  
- Facilitator laptop with presentation loaded and connected to projector.  
- Flip chart or whiteboard and markers for group debrief notes (optional).  

**For Remote Participants:**  
- Digital copies of all worksheets and datasets shared prior to or at the start of the workshop via chat or shared drive link.  
- Access instructions for breakout rooms clearly communicated and supported.  
- Facilitation prompts provided verbally and in chat for each session segment.  
- A virtual timer or verbal cues for time management during exercises.  
- Links or resources for further learning provided digitally at wrap-up.  

---

### 3. Timing and Facilitation Notes

| Time           | Activity                                     | Facilitation Notes                                                                                       |
|----------------|----------------------------------------------|-------------------------------------------------------------------------------------------------------|
| 0:00 - 0:10    | Welcome & Introductions                       | Facilitate quick introductions by name + expectation from each participant (both in room & virtual). Use chat box for remote attendees to quickly share if volume/time is constrained. |
| 0:10 - 0:25    | What is AI? – Plain Language Overview        | Present slides clearly, using analogies and plain language. Check remote attendees’ understanding via quick Q&A or chat prompts. Ensure camera and mic quality are good for virtual attendees. |
| 0:25 - 0:40    | Real-World AI Examples                        | Engage all participants by inviting examples from both in-room and remote attendees. Use shared slide deck, encourage chat interaction. |
| 0:40 - 1:00    | Key AI Concepts Explained Simply             | Speak slowly and clearly; occasionally summarize key points. Utilize visuals on screen. Confirm remote visibility/clarity of slides. |
| 1:00 - 1:10    | Break                                         | Announce break; inform remote attendees they may stretch off-screen but encourage return on time. Use a visible timer or automated break reminder. |
| 1:10 - 1:40    | Hands-On Interactive Exercise: Decision Simulation | Divide participants into small groups (3-5 persons):  
   - In-room groups move to designated tables.  
   - Remote participants assigned to breakout rooms by co-host or facilitator.  
   Facilitator visits each group/breakout room to answer questions and keep discussions on track. Provide clear written and verbal instructions before breakout start. Timer announced at start and reminders at 10- and 5-minute marks. Encourage note-taking on findings for debrief. |
| 1:40 - 1:55    | Group Debrief & Discussion                    | Re-group all participants (in-room and virtual) into main session space. Facilitate sharing of insights; use chat & raise-hand features to manage input from remote attendees. Consider directing questions alternately to in-room and virtual attendees for balance. Record key points on flip chart or shared digital whiteboard. |
| 1:55 - 2:00    | Wrap-Up, Q&A, and Next Steps                  | Provide concise summary, repeat key takeaways. Share digital resource links and contact info in chat. Open floor for final questions from all attendees. Close session with thank you and instructions for feedback survey, if applicable. |

---

### 4. Remote Attendee Support

- Distribute clear instructions before the workshop for joining the video conference, including troubleshooting tips and contacts for IT support.  
- Assign a co-host or technical assistant to manage the chat, invitations to breakout rooms, and to troubleshoot remote participant issues in real time.  
- Prior to breakout exercises, provide a live tutorial or quick demo of how to use breakout rooms and raise-hand/chat functions.  
- Offer alternative communication methods (email or phone) for remote attendees to reach out if they lose connection.  
- During breaks and transitions, remind remote attendees of expected return times and encourage use of video/audio to stay engaged.  
- Allow remote participants to use chat or “raise hand” for questions throughout the session, with facilitator actively monitoring the chat pane.  
- Share all workshop materials well in advance and again at the start via chat to ensure remote participants have easy access.  
- Use a reliable platform with good audio/video quality, encourage participants to use headsets and stable internet connections.

---

### 5. Risks or Setup Issues to Avoid

- **Audio-Visual Issues:**  
  - Avoid insufficient microphone or speaker setup that causes remote attendees to struggle hearing or in-room participants not hearing remote voices.  
  - Test all AV equipment including projector, video conferencing sound, and facilitator laptop before session starts.  
  - Have backup speakers/mic systems ready or use facilitator’s laptop mic if room audio fails.  

- **Connectivity Problems:**  
  - Ensure venue has strong WiFi with adequate bandwidth for streaming and multiple simultaneous video streams.  
  - Prepare alternative dial-in or join by phone options for remote attendees if internet fails.  
  - Avoid excessive simultanous screen sharing or videos that can burden bandwidth.  

- **Breakout Room Confusion:**  
  - Pre-assign breakout rooms or assign them immediately once participants have joined to avoid delays.  
  - Clearly explain breakout procedures verbally and share instructions in chat before breakout start.  
  - Have facilitator and co-host monitor breakout rooms to quickly intervene or assist.  

- **Engagement Imbalance:**  
  - Avoid letting remote attendees feel isolated by regularly including them in questions or asking for their input.  
  - Avoid framing discussions only for in-room participants. Use chat polling or virtual hand raise to include remote voices.  
  - Minimize lecture style; keep session interactive with pauses for Q&A and opportunities for sharing.  

- **Timing Overruns:**  
  - Strictly monitor timing with visible timer and reminders.  
  - Prepare to gently nudge groups to wrap up discussions to keep on schedule.  
  - Have a flexible plan to reduce discussion or Q&A time if running late but keep core content delivered.  

- **Material Access Issues:**  
  - Avoid relying solely on printed materials for remote participants; share all documents digitally well before and at session start.  
  - Verify that digital documents are accessible and formatted for easy viewing on various devices.  

---

**End of Logistics Plan**



# 3. Budget Plan

## Agent
**Budget Analyst**

## Task
Review the curriculum and logistics plans.

Create a budget plan under $1,500.

Your output must include:
1. Estimated cost categories
2. What to spend money on
3. What to avoid spending money on
4. Contingency reserve
5. Any changes needed to keep the plan affordable

Do not create a new workshop.
Constrain and improve the existing plan financially.

## Output
**Budget Plan for "Demystifying AI for Managers" Workshop (Hybrid Delivery)**  
**Total Budget Target: Under $1,500**

---

### 1. Estimated Cost Categories

| Category                     | Estimated Cost | Notes                                                        |
|------------------------------|----------------|--------------------------------------------------------------|
| Venue Rental                 | $300           | Small meeting room with AV infrastructure                     |
| AV Equipment Rental          | $150           | Projector, mic, speakers (only if not included in venue cost)|
| Video Conferencing Platform  | $0 - $100      | Use existing license preferred; otherwise low-cost plan      |
| Printing of Materials        | $40            | Datasets, worksheets — print only for in-person groups        |
| Facilitator Fee             | $600           | Includes design and delivery of workshop                      |
| Miscellaneous Supplies       | $100           | Name tags, pens, notepads, flip chart markers                 |
| Refreshments (optional)      | $150           | Coffee/snacks for in-person attendees (can be reduced or removed) |
| Contingency Reserve          | $100           | For unexpected expenses or last minute tech support          |
| **Total Estimated Budget**  | **$1,440-1,540** | Aiming to keep at or below $1,500 with tradeoffs as below    |

---

### 2. What to Spend Money On

- **Venue Rental:** A modest, well-located room with reliable WiFi and basic AV support to ensure hybrid interaction works smoothly.
- **AV Equipment Rental:** Only if the venue does not provide projector, sound system, and microphone. Emphasize testing beforehand.
- **Facilitator Fee:** Essential for ensuring a high quality, engaging workshop facilitated by someone versed in plain-language AI explanations and hybrid facilitation.
- **Printing Materials:** Limit printing to only what is necessary (worksheet + dataset sheets) and only for in-person participants. Provide digital versions to remote participants.
- **Miscellaneous Supplies:** Name tags improve engagement, and pens/notepads aid note-taking. A flip chart can be borrowed from venue or facilitator to avoid extra cost.
- **Refreshments:** Provide coffee and light snacks to keep cognitive engagement high if budget permits, but optional.
- **Video Conferencing Platform License:** Prefer using an existing license or free plans that suffice (Zoom free allows breakout rooms with limitations). Consider small upgrade if necessary.
- **Contingency Reserve:** Always keep reserve for unforeseen issues such as last-minute printing, additional supplies, or tech fixes.

---

### 3. What to Avoid Spending Money On

- **Excessive Venue Costs:** Avoid large conference halls or premium hotels. Pick small business centers or community meeting rooms.
- **Advanced AV Rentals:** Avoid high-end recording or streaming equipment not necessary for a 2-hour workshop.
- **Excess Printing:** Do not print participant guides or materials for all participants; remote attendees get digital-only copies.
- **Elaborate Swag or Gifts:** Avoid branded giveaways, elaborate gift bags, or expensive promotional items.
- **Catering Full Meals:** Limit food to light refreshments or skip entirely if budget-constrained.
- **Multiple Facilitators:** A single well-prepared facilitator suffices; co-facilitators add cost.
- **Premium Video Conferencing Plans:** Avoid expensive enterprise licenses if breakout rooms and screen share are available in free/low-cost options.

---

### 4. Contingency Reserve

- **Amount:** $100 (about 7% of total budget)
- **Purpose:**  
  - Technical hiccups (e.g., backup printing, last-minute software purchase)  
  - Unexpected supply needs (e.g., extra markers, thumb drives)  
  - Refreshment shortfalls  
  - Minor overages in venue or AV costs

---

### 5. Recommendations & Changes to Keep Plan Affordable

- **Venue and AV:**  
  - Confirm if venue includes projector and audio system; if so, avoid rental costs.  
  - Consider smaller venues, or co-working spaces with meeting rooms at reduced rates.  
  - Check if organization already has a video conferencing license with breakout rooms to avoid new software costs.  
  - Confirm WiFi capability prior to booking.

- **Printing:**  
  - Reduce pages per participant by consolidating worksheet and dataset onto fewer pages.  
  - Print on duplex (double-sided) and use economical paper.  
  - Only supply printed packets to in-person groups; remote participants get only digital files (shared before or at start).

- **Materials Supply:**  
  - Use reusable name tags and pens if possible, or print simple paper badges onsite.  
  - Borrow flip charts or whiteboards from venue to avoid rental/purchase.

- **Refreshments:**  
  - If budget constrained, switch to providing water and coffee only; skip snacks.  
  - Alternatively, ask participants to bring their own refreshments or keep the break informal.

- **Facilitator Fee:**  
  - Confirm facilitator’s rate includes preparation and delivery only, avoiding additional overhead.  
  - Consider partial remote delivery by facilitator (e.g., hybrid hosting) to reduce travel costs if any.

- **Technology Backup:**  
  - Use free timer apps or features integrated in conferencing platform rather than purchasing devices.  
  - Test all tech ahead, reducing risk of unplanned expenses for alternatives.

- **Hybrid Setup Efficiency:**  
  - Clearly prepare digital instructions and materials shared via chat or drive links to minimize printing and support time.  
  - Assign a technical assistant only if needed and budget allows, otherwise facilitator handles tech with straightforward platform tools.

- **Timing Management:**  
  - Strictly adhere to agenda to avoid overtime venue or facilitator fees.

---

### Final Adjusted Budget Estimate (Sample Model)

| Item                            | Cost     | Notes                                   |
|--------------------------------|----------|-----------------------------------------|
| Venue Rental                   | $300     | Small business meeting room              |
| AV Equipment                  | $0 - $150| Use venue tech if included, else rent    |
| Video Conferencing Platform   | $0       | Use existing Zoom/MS Teams license       |
| Printing Materials            | $40      | Minimal, in-person only                   |
| Facilitator Fee               | $600     | Fixed for design and delivery             |
| Miscellaneous Supplies       | $80      | Name tags, pens, markers                  |
| Refreshments (optional)       | $120     | Light snacks & coffee or water only      |
| Contingency                  | $100     | Reserve for unforeseen expenses          |
| **Total Estimated Cost**      | **$1,240-$1,390** | Under budget with room for flexibility |

---

**Summary:**  
By leveraging existing technology licenses, minimizing printed materials, selecting a cost-efficient venue with AV included, and scaling back refreshments and misc supplies, the workshop budget can be responsibly set under $1,500. A contingency reserve buffers against surprises. Avoid extras that do not directly support the learning experience or engagement, thus delivering a practical, effective workshop within financial constraints.  

---

**End of Budget Plan**



# 4. Final Coordinated Workshop Plan

## Agent
**Workshop Coordinator**

## Task
Use the curriculum, logistics, and budget work to create the final workshop plan.

Your output must include:
1. Workshop title
2. Learning objectives
3. Final 2-hour agenda
4. Hands-on exercise
5. Logistics checklist
6. Budget summary
7. Facilitator preparation checklist

Make the final plan clear enough that an event organizer could use it.

## Output
---
# Final Workshop Plan:  
**Workshop Title:**  
Demystifying AI for Managers: Practical Insights and Hands-On Experience  

---

## 1. Learning Objectives  
By the end of this 2-hour workshop, participants will be able to:  
1. Define Artificial Intelligence (AI) in simple, practical terms.  
2. Recognize common AI applications relevant to their business context.  
3. Understand key AI concepts without technical jargon (e.g., machine learning, data training).  
4. Identify opportunities and challenges related to AI in management decisions.  
5. Collaborate through an interactive exercise that simulates AI-powered decision-making.  
6. Engage in meaningful conversations about integrating AI thoughtfully into their teams and workflows.

---

## 2. Final 2-Hour Agenda

| Time         | Activity                                    | Format                      | Notes                                                      |
|--------------|---------------------------------------------|-----------------------------|------------------------------------------------------------|
| 0:00 – 0:10  | **Welcome & Introductions**                  | Whole group (in-person + virtual) | Quick introductions: name + one expectation from each participant; establish rapport |
| 0:10 – 0:25  | **What is AI? – Plain Language Overview**   | Facilitator presentation + Q&A  | Use analogies, avoid jargon, check understanding           |
| 0:25 – 0:40  | **Real-World AI Examples**                    | Group discussion + slides         | Focus on industries/tasks familiar to participants         |
| 0:40 – 1:00  | **Key AI Concepts Explained Simply**         | Facilitator explanation            | Concepts covered: Machine Learning, Data Training, Algorithms, Automation |
| 1:00 – 1:10  | **Break**                                    | Break                         | Stretch/stretch break for all attendees                     |
| 1:10 – 1:40  | **Hands-On Exercise: AI in Action – Decision Simulation** | Small groups / breakout rooms | Groups analyze dataset, “train” AI, make decisions, discuss bias |
| 1:40 – 1:55  | **Group Debrief & Discussion**                | Whole group discussion            | Share insights, reflect on exercise outcomes                |
| 1:55 – 2:00  | **Wrap-Up, Q&A, Next Steps**                   | Facilitator-led discussion          | Summarize key takeaways, provide resource links, final questions |

---

## 3. Hands-On Exercise  

**Title:** AI in Action – Decision Simulation  

**Objective:**  
To experience how AI systems use data to make recommendations and understand implications for decision-making.

**Setup:**  
- Participants divided into groups of 3–5 people. For virtual attendees, use breakout rooms.  
- Each group receives a simple dataset (printed or digital) — e.g., customer age, purchase history, satisfaction rating.  
- Groups receive a "decision challenge" such as selecting a marketing channel or task assignment.

**Instructions:**  
1. Review data in group.  
2. Simulate “training” an AI by identifying key patterns/trends.  
3. Make a recommendation based on data analysis.  
4. Discuss possible biases or gaps in the data affecting decisions.

**Materials:**  
- Sample dataset (one per group)  
- Worksheet with guiding prompts:  
  - What patterns do you observe?  
  - What decision would the AI recommend?  
  - What potential biases or missing info may affect outcomes?  
- Timer (visible or announced)

**Facilitator Role:**  
- Provide instruction and clarify exercise steps.  
- Support and guide group discussions, monitor breakout rooms.  
- Encourage reflection on AI learning process and human oversight.

---

## 4. Logistics Checklist  

**Venue (In-Person) Setup:**  
- Room arranged with small tables (3-5 seats) for group exercises.  
- Projector and screen/large monitor visible to all.  
- Strong WiFi capable of video conferencing for hybrid participation.  
- Microphone and speaker system, if room size requires.  
- Name tags pre-placed on tables.  
- Printed copies of datasets and worksheets for each group.  
- Timer (physical or app).  
- Pens and notepads for participants.  
- Flip chart or whiteboard with markers for debrief notes (optional).

**Virtual Setup:**  
- Video conferencing platform with breakout room capability (Zoom, MS Teams, etc.).  
- Breakout rooms pre-configured or facilitator assigns dynamically.  
- Shared drive or chat link with presentation slides, datasets, and worksheets distributed at start.  
- Co-host or technical assistant to support chat/questions and breakout coordination.  
- Clear instructions for joining, using breakout rooms, and chat/Q&A functionalities.  
- Virtual timer or verbal timing cues.

**Technology:**  
- Facilitator laptop with loaded presentations.  
- Stable internet connection.  
- Backup plan if breakout functionality fails (e.g., all participants remain in main room for discussion or virtual polling).

**Materials:**  
- Printed or digital workshop agenda distributed to participants.  
- Dataset and worksheet packets for groups.  
- Name tags for in-person attendees.  

---

## 5. Budget Summary (Estimated)

| Item                               | Estimated Cost | Notes                                               |
|-----------------------------------|----------------|-----------------------------------------------------|
| Venue Rental                      | $300           | Small meeting room with WiFi and AV equipment       |
| AV Equipment Rental (if needed)   | $0–$150        | Projector, microphone, speakers—if not provided     |
| Video Conferencing Platform        | $0             | Using existing license (Zoom/MS Teams free tier)    |
| Printing Handouts and Datasets     | $40            | For in-person groups only                            |
| Facilitator Fee                   | $600           | Preparation and delivery of workshop                 |
| Miscellaneous Supplies (pens, tags) | $80           | Name tags, flip chart markers, pens                   |
| Refreshments (optional)            | $120           | Light snacks and coffee for in-person attendees      |
| Contingency                      | $100           | For unexpected expenses                               |
| **Total Estimated Budget**         | **$1,240-$1,390** | Contingency included, budget under $1,500            |

*Notes:*  
- Virtual delivery or fully remote workshop reduces venue and AV costs significantly.  
- Printing limited to in-person participants minimizes costs.

---

## 6. Facilitator Preparation Checklist   

- Review participant business contexts to tailor examples appropriately.  
- Master explanations of AI concepts in plain language; practice analogies.  
- Familiarize thoroughly with slide deck and materials in advance.  
- Test technology (projector, mic, conferencing platform, breakout rooms) before workshop start.  
- Prepare clear instructions for breakout exercise and alternative activities if technical issues arise.  
- Anticipate common questions, especially on AI ethics, bias, and managerial implications; prepare accessible answers.  
- Plan engagement strategies for hybrid audience to include both in-person and virtual participants equally.  
- Review timing, keeping strict watch on agenda to ensure all activities get adequate attention.  
- Prepare resource links and follow-up materials to share at workshop close.  
- Coordinate with technical assistant or co-host for managing breakout rooms and chat during session.  
- Arrange printed materials, name tags, and setup logistics with event organizer before event day.

---

# Summary - Ready-to-Use Workshop Plan for Event Organizer

- Provide facilitator with agenda, materials, budget, logistics checklist, and prep notes above.  
- Confirm venue reservation with specified AV and seating setup.  
- Ensure digital files for slides, datasets, and worksheets are uploaded and accessible to all participants before workshop.  
- Distribute printed materials, name tags, and supplies at venue setup time.  
- Confirm video conferencing setup and conduct a tech check with facilitator and co-host prior to event start.  
- Allocate budget accordingly, keep receipts and contingency available.  
- Facilitate participant sign-in and provide a warm welcome, then follow time allocations carefully.

---

**End of Final Workshop Plan**


## Comparison - single agent vs Crew
We'll build CrewAI agents to evaluate the two approaches to event planning

In [9]:
from IPython.display import display, Markdown

comparison_prompt = f"""
Compare the single-agent baseline against the CrewAI final workshop plan.

Single-agent baseline:
{task_output_text(baseline_task)}

CrewAI final plan:
{task_output_text(final_plan_task)}

Create a concise side-by-side comparison table with these rows:
1. Learning objectives
2. Agenda structure
3. Hands-on exercise
4. Logistics readiness
5. Budget realism
6. Facilitator prep
7. Overall usefulness

For each row, include:
- what the single-agent version did
- what the CrewAI version did
- which version is stronger and why

Be specific and avoid generic praise.
"""

comparison_agent = Agent(
    role="Comparison Analyst",
    goal="Compare two workshop plans clearly and fairly.",
    backstory="You evaluate planning documents and identify practical differences.",
    llm=MODEL_NAME,
    verbose=False,
)

comparison_task = Task(
    description=comparison_prompt,
    expected_output="A concise side-by-side comparison table.",
    agent=comparison_agent,
)

comparison_crew = Crew(
    agents=[comparison_agent],
    tasks=[comparison_task],
    process=Process.sequential,
    verbose=False,
)

comparison_result = comparison_crew.kickoff()

display(Markdown(comparison_task.output.raw))

| Aspect               | Single-Agent Baseline                                             | CrewAI Final Plan                                                 | Stronger Version & Rationale                                                                                                 |
|----------------------|------------------------------------------------------------------|------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------|
| **1. Learning Objectives** | Focused on understanding AI basics, business applications, limitations, ethics, and basic tool application for managers. Included confidence in discussing AI with tech teams. | Defined clear, practical AI understanding using plain language including AI concepts, data training, bias, plus collaboration and thoughtful AI integration in teams. | **CrewAI**: More comprehensive and nuanced objectives including understanding bias and AI concepts, plus fostering collaboration and deliberate integration.     |
| **2. Agenda Structure**    | 2 hours split into intro, AI definitions, use cases, ethics, short break, 40-minute hands-on exercise with AI tool, wrap-up. Mix of presentations, discussions, group exercises. | 2 hours with similar timing but explicitly includes explanation of key concepts like machine learning, data training; longer group debrief after exercise; well-timed activities. | **CrewAI**: More detailed coverage of AI concepts, explicit inclusion of debrief to reinforce learning, more structured pacing around key content.                 |
| **3. Hands-On Exercise**    | Simple AI tools like text summarization or sentiment analysis applied on sample data; group work with facilitator support; focus on data input and interpretation. | Simulated AI decision-making by “training” AI with dataset analysis, making recommendations, discussing bias; guided worksheets used; facilitator supports deeper reflection. | **CrewAI**: More interactive and reflective exercise with structured worksheet prompts, aligns better with learning about AI decision impacts and biases.            |
| **4. Logistics Readiness** | Basic checklist includes venue, AV, Wi-Fi, participant devices, remote setup, pre-register AI tools, printed materials, optional refreshments, registration. | More detailed logistics covering seating arrangement, name tags, printed materials, timer tools, flip charts, breakout setup, co-host tech support, hybrid engagement. | **CrewAI**: More thorough and practical logistics preparation, especially for hybrid settings and participant engagement with defined roles and materials.           |
| **5. Budget Realism**       | Estimated $1,100–$1,500; includes facilitator fee, venue, AV, printing, refreshments; some costs zero if internal resources used; clear contingency. | Slightly more detailed budget $1,240–$1,390; accounts for printing limited to in-person, explicit contingency, facilitator fee lower, includes name tags, supplies. | **CrewAI**: More precise cost allocation tied to delivery format, with realistic contingencies and practical inclusions like name tags and supplies.                  |
| **6. Facilitator Prep**     | Emphasizes tool familiarity, tech testing, jargon-free materials, prep datasets, instructions sent before, active facilitation, troubleshooting, post workshop follow-up. | Includes tailored business context review, mastering simple explanations, tech and breakout prep, anticipated questions, hybrid engagement strategies, co-host coordination. | **CrewAI**: More comprehensive facilitator guidance including audience tailoring, hybrid inclusion, question anticipation, and collaboration with technical support.  |
| **7. Overall Usefulness**   | Provides a solid foundation for non-technical managers to learn AI basics, interact with AI tools, and discuss AI with technical teams; suitable for straightforward delivery. | Offers a richer, more interactive and reflective experience; better accommodates hybrid delivery; promotes deeper understanding of AI implications and team integration. | **CrewAI**: Stronger overall due to depth of content, detailed facilitation strategy, hybrid readiness, and promoting thoughtful AI adoption beyond basics.             |

## Why Did CrewAI Improve the Result?

The CrewAI version did not improve because it used "more AI."

It improved because:
- the problem decomposed naturally into specialized responsibilities
- each agent focused deeply on a narrower concern
- later tasks refined and operationalized earlier work
- the coordinator integrated multiple specialized perspectives

This is one of the key strengths of role-oriented multi-agent systems.

However, the improvement also came with costs:
- more orchestration
- more prompts
- more token usage
- more implementation complexity

Multi-agent systems are most valuable when:
- the task naturally decomposes
- specialization adds meaningful perspective
- handoffs improve the final artifact


## What This Demo Should Show

This is a basic CrewAI use case.

The value comes from:
- specialized roles
- explicit tasks
- ordered handoffs
- structured final output

This is different from the AutoGen demos.

AutoGen emphasized interaction and conversation.

CrewAI emphasizes delegated work and coordinated task completion.

---

## Bridge to the Next Demo

This workshop demo uses one simple pass through the crew.

The next demo, the bike-shop planning crew, adds:
- stronger business constraints
- operational tradeoffs
- financial pressure
- a second refinement cycle
- a more mature final recommendation

So the progression is:

1. Workshop demo: task delegation and handoffs
2. Bike-shop demo: iterative refinement under constraints
